# FPS Performance Analysis
Visualisation of per-frame processing latency and throughput from the pose inference pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ---------- Publication-quality style ----------
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,
})

# Nature / Science inspired palette (Tol bright)
C_BLUE   = '#4477AA'
C_CYAN   = '#66CCEE'
C_GREEN  = '#228833'
C_YELLOW = '#CCBB44'
C_RED    = '#EE6677'
C_PURPLE = '#AA3377'
C_GREY   = '#BBBBBB'

# ---------- Load data ----------
df = pd.read_csv('../fps_metrics.csv')
print(f'Loaded {len(df)} frames')
df.head()

In [ ]:
# ---------- Summary statistics ----------
stats = df[['elapsed_ms', 'fps']].describe().round(2)
print('\n--- Performance Summary ---')
print(f"Mean latency : {df['elapsed_ms'].mean():.2f} ms")
print(f"Median latency: {df['elapsed_ms'].median():.2f} ms")
print(f"Std latency  : {df['elapsed_ms'].std():.2f} ms")
print(f"Mean FPS     : {df['fps'].mean():.2f}")
print(f"Median FPS   : {df['fps'].median():.2f}")
print(f"Min FPS      : {df['fps'].min():.2f}")
print(f"Max FPS      : {df['fps'].max():.2f}")
stats

In [ ]:
# ---------- Plot 1: FPS over time ----------
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df['frame_index'], df['fps'], color=C_BLUE, linewidth=0.8, alpha=0.7, label='Instantaneous FPS')

# Rolling average (window = 10)
window = min(10, len(df))
rolling_fps = df['fps'].rolling(window=window, center=True).mean()
ax.plot(df['frame_index'], rolling_fps, color=C_RED, linewidth=2, label=f'Rolling avg (w={window})')

# Mean line
mean_fps = df['fps'].mean()
ax.axhline(mean_fps, color=C_GREEN, linestyle='--', linewidth=1.2, alpha=0.8, label=f'Mean = {mean_fps:.1f} FPS')

ax.set_xlabel('Frame Index')
ax.set_ylabel('FPS')
ax.set_title('Instantaneous FPS per Frame')
ax.legend(frameon=False)
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('fps_over_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---------- Plot 2: Latency over time ----------
fig, ax = plt.subplots(figsize=(10, 4))

ax.fill_between(df['frame_index'], df['elapsed_ms'], color=C_CYAN, alpha=0.3)
ax.plot(df['frame_index'], df['elapsed_ms'], color=C_BLUE, linewidth=0.8, alpha=0.8, label='Frame latency')

rolling_lat = df['elapsed_ms'].rolling(window=window, center=True).mean()
ax.plot(df['frame_index'], rolling_lat, color=C_RED, linewidth=2, label=f'Rolling avg (w={window})')

mean_lat = df['elapsed_ms'].mean()
ax.axhline(mean_lat, color=C_GREEN, linestyle='--', linewidth=1.2, alpha=0.8, label=f'Mean = {mean_lat:.1f} ms')

ax.set_xlabel('Frame Index')
ax.set_ylabel('Latency (ms)')
ax.set_title('Per-Frame Processing Latency')
ax.legend(frameon=False)
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('latency_over_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---------- Plot 3: Latency distribution ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Histogram
ax1.hist(df['elapsed_ms'], bins=30, color=C_BLUE, alpha=0.7, edgecolor='white', linewidth=0.5)
ax1.axvline(df['elapsed_ms'].median(), color=C_RED, linestyle='--', linewidth=1.5, label=f'Median = {df["elapsed_ms"].median():.1f} ms')
p95 = df['elapsed_ms'].quantile(0.95)
ax1.axvline(p95, color=C_PURPLE, linestyle=':', linewidth=1.5, label=f'P95 = {p95:.1f} ms')
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Count')
ax1.set_title('Latency Distribution')
ax1.legend(frameon=False)

# Box plot
bp = ax2.boxplot([df['elapsed_ms'], df['fps']], labels=['Latency (ms)', 'FPS'],
                 patch_artist=True, widths=0.5,
                 boxprops=dict(facecolor=C_CYAN, alpha=0.5),
                 medianprops=dict(color=C_RED, linewidth=2),
                 whiskerprops=dict(color=C_BLUE),
                 capprops=dict(color=C_BLUE),
                 flierprops=dict(marker='o', markerfacecolor=C_GREY, markersize=4, alpha=0.5))
ax2.set_title('Latency & FPS Distribution')

plt.tight_layout()
plt.savefig('latency_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---------- Plot 4: Cumulative latency ----------
fig, ax = plt.subplots(figsize=(10, 4))

cumulative_ms = df['elapsed_ms'].cumsum() / 1000  # seconds
ax.plot(df['frame_index'], cumulative_ms, color=C_PURPLE, linewidth=2)
ax.fill_between(df['frame_index'], cumulative_ms, color=C_PURPLE, alpha=0.1)

ax.set_xlabel('Frame Index')
ax.set_ylabel('Cumulative Time (s)')
ax.set_title('Cumulative Processing Time')
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('cumulative_time.png', bbox_inches='tight')
plt.show()

print(f'\nTotal processing time: {cumulative_ms.iloc[-1]:.2f} s for {len(df)} frames')
print(f'Effective throughput : {len(df) / cumulative_ms.iloc[-1]:.2f} FPS')